In [21]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df["review"]=df["review"].str.lower()
df["sentiment"] = df["sentiment"].map({"positive":1, "negative":0})

In [5]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. <br /><br />the...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [6]:
df.shape

(50000, 2)

In [7]:
train, test = train_test_split(df, test_size = 0.2, random_state=42)

In [8]:
train_words = [review.split() for review in train["review"]]
test_words = [review.split() for review in test["review"]]

In [9]:
w2v = Word2Vec(
    sentences = train_words,
    vector_size = 50,
    window = 2,
    min_count = 2,
    sg = 0,
    epochs = 5
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [17]:
def document_vector(words):
    vectors=[
        w2v.wv[w] 
        for w in words 
        if w in w2v.wv]
    return np.mean(vectors, axis = 0)

In [18]:
X_train = np.array([
    document_vector(x)
    for x in train_words
])

X_test = np.array([
    document_vector(x)
    for x in test_words
])

In [28]:
y_train = train["sentiment"].values
y_test = test["sentiment"].values

In [22]:
model = Sequential([
    Input(shape=(50, )),
    Dense(32, activation = "relu"),
    Dense(1, activation = "sigmoid")
])

In [23]:
model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

In [25]:
model.fit(
    X_train, 
    y_train, 
    epochs = 10, 
    batch_size = 32
)

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 917us/step - accuracy: 0.7698 - loss: 0.4844
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 905us/step - accuracy: 0.7702 - loss: 0.4828
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 883us/step - accuracy: 0.7721 - loss: 0.4810
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 930us/step - accuracy: 0.7730 - loss: 0.4798
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 985us/step - accuracy: 0.7723 - loss: 0.4786
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 928us/step - accuracy: 0.7721 - loss: 0.4772
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 913us/step - accuracy: 0.7751 - loss: 0.4754
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 935us/step - accuracy: 0.7758 - loss: 0.4745
Epoch 9/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 903us/step - accuracy: 0.7760 - loss: 0.4737
Epoch 10/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 909us/step - accuracy: 0.7763 - loss: 0.4741


In [29]:
loss, accuracy = model.evaluate(X_test, y_test)
print("loss: ", loss)
print("accuracy: ", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 841us/step - accuracy: 0.7727 - loss: 0.4764 
loss:  0.4763544797897339
accuracy:  0.7727000117301941
